In [ ]:
# Fabric Notebook: NB_03_Audit_Log_Failed
# Called inside ForEach on the FAILED path of Copy_SFTP_to_Bronze.
# ============================================================
# HOW PARAMETERS WORK IN FABRIC:
# Cell 1 must be tagged as a Parameters cell.
# To tag it: click "..." on Cell 1 → "Toggle parameter cell"
# A "Parameters" badge appears on the cell.
# The pipeline overrides these default values at runtime.
# ============================================================

# ── CELL 1 — TAG THIS CELL AS "Parameters" ──────────────────
# Click "..." on this cell → "Toggle parameter cell"
p_pipeline_run_id = "manual_run"
p_file_name       = "unknown_file.csv"
p_error_message   = "unknown error"


In [ ]:

# ── CELL 2 — Main logic ──────────────────────────────────────
from pyspark.sql import SparkSession
from datetime import datetime, timezone

spark = SparkSession.builder.getOrCreate()

print(f"PipelineRunId : {p_pipeline_run_id}")
print(f"FileName      : {p_file_name}")
print(f"Error         : {p_error_message}")

AUDIT_TABLE = "bronze_audit_log"

now_utc  = datetime.now(timezone.utc)
audit_id = int(now_utc.timestamp() * 1000)
now_str  = now_utc.strftime("%Y-%m-%d %H:%M:%S")
err_safe = p_error_message.replace("'", "''") if p_error_message else "Unknown error"

spark.sql(f"""
    INSERT INTO {AUDIT_TABLE}
        (AuditId, PipelineRunId, FileName, FileModifiedTime,
         DataLoadedDate, BronzePath, Status, ErrorMessage)
    VALUES (
        {audit_id},
        '{p_pipeline_run_id}',
        '{p_file_name}',
        NULL,
        CAST('{now_str}' AS TIMESTAMP),
        NULL,
        'Failed',
        '{err_safe}'
    )
""")

print(f"  Audit row inserted — {p_file_name} → Failed")
print(f"  File remains in SFTP Upload folder for retry on next run.")